# Optimización de controles operativos en minería subterránea con clasificación multietiqueta

**Contexto del caso:**
- Una operación minera subterránea registra, tarea por tarea, si cada control de seguridad y calidad se cumplió o no. Cada control es una pregunta de sí o no: ¿se hizo la revisión de sostenimiento?, ¿se verificó la ventilación?, ¿se registró la voladura correctamente?
- El objetivo real no era predecir por predecir: era optimizar la operación. ¿Qué operadores tienen mejores resultados? ¿Qué controles se pueden simplificar? ¿Cuáles necesitan más atención?
- Cada tarea se agrupa por tipo de roca y tipo de labor en 4 grupos operativos, porque el comportamiento de los controles cambia según el contexto geológico y el tipo de trabajo.
- Artículo completo, con el diagrama del pipeline: **https://fuzzyfrog.ai/es/ai-lab/proyectos/mineria/optimizacion-controles-operativos-mineria-machine-learning/**

**Nota sobre los datos:** este notebook usa datos de ejemplo con la misma estructura de columnas que el proyecto original (una operación minera subterránea real, usada con autorización para fines de tesis). Sustituye `data/controles_operativos.csv` por tus propios registros manteniendo el mismo esquema.


## Diagrama del pipeline

```
Registros de tareas (por operador, zona, guardia, roca, labor)
                    │
        Agrupar por tipo de roca + tipo de labor
        (4 grupos: distinto comportamiento geológico)
                    │
        EDA: cumplimiento histórico por roca, labor, zona, guardia, operador
                    │
        Modelo multietiqueta
        (AVANCE → predice todos los controles binarios a la vez)
                    │
        ┌───────────┼───────────┬────────────┬──────────┐
        │           │           │            │          │
   Regresión    Random      SVC          KNN      Naive Bayes
   Logística    Forest
        │           │           │            │          │
        └───────────┼───────────┴────────────┴──────────┘
                    │
        Evaluación en 2 capas:
        métricas clásicas (accuracy, F1, Hamming loss)
        + matriz de error por control (dónde y cómo se equivoca)
```

El diagrama interactivo con tooltip por bloque está embebido en el artículo de la plataforma.


## 1. Carga de datos

- Cada fila es una tarea: fecha, zona, guardia, operador, tipo de roca, tipo de labor, avance y un conjunto de columnas binarias, una por control de seguridad/calidad.
- Se eliminan columnas administrativas que no aportan al agrupamiento (turno detallado, jefe de guardia, longitud de perforación, nivel, sección).


In [ ]:
import pandas as pd

data = pd.read_csv("data/controles_operativos.csv", parse_dates=["FECHA"])

columnas_administrativas = ["TURNO", "JEFE_DE_GUARDIA", "LONGITUD_DE_PERFORACION", "NIVEL", "SECCION"]
data = data.drop(columns=[c for c in columnas_administrativas if c in data.columns])
data = data.dropna()

print(f"Tareas registradas: {len(data)}")
data.head(3)


## 2. Explicación de los datos: variables predictoras binarias

**Este es el insight central del proyecto.** La mayoría de las variables predictoras aquí no son continuas, son binarias: 1 si el control se realizó, 0 si no. Esto cambia el tratamiento en dos frentes:

- No tiene sentido tratarlas como variables numéricas normales para estadística descriptiva; hay que resumirlas como tasas de cumplimiento (proporción de 1s), no como promedios sin más contexto.
- Cuando son la variable objetivo (como en este proyecto, donde se predicen varios controles a la vez), se necesita clasificación multietiqueta, no una clasificación binaria simple repetida sin cuidado.


In [ ]:
columnas_booleanas = [c for c in data.columns if set(data[c].dropna().unique()).issubset({0, 1})]
print(f"Controles binarios detectados: {len(columnas_booleanas)}")

tasa_cumplimiento = data[columnas_booleanas].mean().sort_values()
tasa_cumplimiento.tail(10)


## 3. Ingeniería de variables: agrupar por tipo de roca y tipo de labor

Condición del proyecto: el comportamiento de los controles cambia según el contexto geológico y el tipo de trabajo. Modelar todas las tareas juntas habría mezclado patrones que en realidad pertenecen a situaciones operativas distintas.


In [ ]:
condiciones_grupos = {
    "grupo_I":   (data["TIPO_ROCA"].isin(["IIIA", "IIIB"])) & (data["TIPO_LABOR"].isin(["GL", "SN"])),
    "grupo_II":  (data["TIPO_ROCA"].isin(["IIIA", "IIIB"])) & (~data["TIPO_LABOR"].isin(["GL", "SN"])),
    "grupo_III": (data["TIPO_ROCA"].isin(["IVA", "IVB"]))  & (data["TIPO_LABOR"].isin(["GL", "SN"])),
    "grupo_IV":  (data["TIPO_ROCA"].isin(["IVA", "IVB"]))  & (~data["TIPO_LABOR"].isin(["GL", "SN"])),
}

titulos_grupos = {
    "grupo_I": "Roca III (A/B) - Mineral", "grupo_II": "Roca III (A/B) - Desmonte",
    "grupo_III": "Roca IV (A/B) - Mineral", "grupo_IV": "Roca IV (A/B) - Desmonte",
}

for nombre, condicion in condiciones_grupos.items():
    print(f"{titulos_grupos[nombre]}: {condicion.sum()} tareas")


## 4. Análisis exploratorio: cumplimiento por operador

La pregunta de negocio detrás de esto: ¿qué operadores tienen mejores resultados, y en qué controles específicos se puede incidir para mejorar la operación completa?


In [ ]:
import matplotlib.pyplot as plt

cumplimiento_por_operador = (
    data.groupby("OPERADOR_DE_JUMBO")[columnas_booleanas]
    .mean()
    .mean(axis=1)
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
cumplimiento_por_operador.plot(kind="bar", ax=ax, color="#006a87")
ax.set_title("Cumplimiento promedio de controles por operador")
ax.set_ylabel("Proporción de controles cumplidos")
plt.tight_layout()
plt.show()


## 5. Modelado: clasificación multietiqueta

Entrada: `AVANCE` (el ritmo de avance de la tarea). Salida: todos los controles binarios a la vez, como un vector, no uno por uno. Se comparan cinco algoritmos base dentro de un `MultiOutputClassifier`, por cada uno de los 4 grupos operativos.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, hamming_loss

def entrenar_modelo(data_grupo, modelo_base):
    X = data_grupo[["AVANCE"]]
    y = data_grupo[columnas_booleanas]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Si una columna queda con una sola clase en entrenamiento, el clasificador no puede aprenderla;
    # se fuerza al menos un ejemplo de la clase contraria para evitar que el modelo falle en silencio.
    for columna in y_train.columns:
        if y_train[columna].nunique() == 1:
            y_train.iloc[0, y_train.columns.get_loc(columna)] = 1 - y_train.iloc[0, y_train.columns.get_loc(columna)]

    modelo = MultiOutputClassifier(modelo_base, n_jobs=-1)
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    metricas = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=1),
        "hamming_loss": hamming_loss(y_test, y_pred),
    }
    return {"modelo": modelo, "y_test": y_test, "y_pred": y_pred, "metricas": metricas}

modelos_base = {
    "Regresión logística": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVC": SVC(),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
}


In [ ]:
resultados_por_grupo = {}

for nombre_grupo, condicion in condiciones_grupos.items():
    data_grupo = data[condicion][["AVANCE"] + columnas_booleanas].dropna()
    resultados_por_grupo[nombre_grupo] = {}
    for nombre_modelo, modelo_base in modelos_base.items():
        resultado = entrenar_modelo(data_grupo, modelo_base)
        resultados_por_grupo[nombre_grupo][nombre_modelo] = resultado["metricas"]

import pandas as pd
tabla_resultados = pd.DataFrame({
    (grupo, modelo): metricas
    for grupo, modelos in resultados_por_grupo.items()
    for modelo, metricas in modelos.items()
}).T
tabla_resultados


## 6. Evaluación en dos capas

**Segundo insight central del proyecto.** Las métricas clásicas (accuracy, F1, Hamming loss) dicen qué tan bien acertó el modelo en general, pero no dicen *dónde* ni *cómo* se equivocó, y en un contexto operativo eso importa más que el número global: un falso negativo en un control de seguridad no cuesta lo mismo que un falso negativo en un control administrativo. Por eso se agrega una segunda capa de evaluación: una matriz de error por control específico.


In [ ]:
import numpy as np

def matriz_error_por_control(y_test, y_pred, columnas):
    filas = []
    y_pred_df = pd.DataFrame(y_pred, columns=columnas, index=y_test.index)
    for columna in columnas:
        vt = y_test[columna]
        vp = y_pred_df[columna]
        falsos_positivos = ((vt == 0) & (vp == 1)).sum()
        falsos_negativos = ((vt == 1) & (vp == 0)).sum()
        aciertos = (vt == vp).sum()
        filas.append({
            "control": columna,
            "aciertos": aciertos,
            "falsos_positivos": falsos_positivos,
            "falsos_negativos": falsos_negativos,
        })
    return pd.DataFrame(filas).sort_values("falsos_negativos", ascending=False)

# Ejemplo con el mejor modelo del grupo I según Hamming loss
mejor_modelo_grupo_I = min(resultados_por_grupo["grupo_I"], key=lambda m: resultados_por_grupo["grupo_I"][m]["hamming_loss"])
print(f"Mejor modelo para Grupo I: {mejor_modelo_grupo_I}")

data_grupo_I = data[condiciones_grupos["grupo_I"]][["AVANCE"] + columnas_booleanas].dropna()
resultado_grupo_I = entrenar_modelo(data_grupo_I, modelos_base[mejor_modelo_grupo_I])
matriz_error_por_control(resultado_grupo_I["y_test"], resultado_grupo_I["y_pred"], columnas_booleanas).head(10)


## 7. Hallazgos principales

- Tratar los controles como variables binarias, no como numéricas genéricas, cambió tanto el análisis descriptivo (tasas de cumplimiento en vez de promedios) como el modelado (multietiqueta en vez de clasificación simple repetida).
- Ningún algoritmo base ganó en los cuatro grupos operativos: el mejor modelo cambia según el tipo de roca y labor, lo que confirma que agrupar antes de modelar fue la decisión correcta.
- La matriz de error por control reveló que los falsos negativos se concentran en un subconjunto pequeño de controles, justo los que conviene revisar primero con el equipo de operaciones, en vez de intentar mejorar los 30 controles por igual.
- Este tipo de proyecto es un buen punto de partida si trabajas en una operación con datos similares: se puede hacer con datos de tu propia empresa para una tesis, siempre que cuentes con la autorización correspondiente para usarlos.
